In [ ]:
import pandas as pd

# 1. Ingest: load data (example dataset, with anonymized customer identities)
df = pd.read_csv('sales_example.csv', sep=';', encoding='utf-8', on_bad_lines='skip')

# 2. Transform: fix Tiendanube's data inheritance on multi-product orders
# (forward-fills the empty gaps when a customer buys several products in the same order)
columns_to_fill = ['Email', 'Fecha', 'Estado del pago', 'Total']
df[columns_to_fill] = df[columns_to_fill].ffill()

# 3. Clean: isolate real sales (money in the register) and format the timeline
# We only care about rows where 'Estado del pago' (payment status) is 'Recibido' (received)
df_sales = df[df['Estado del pago'] == 'Recibido'].copy()
df_sales['Fecha'] = pd.to_datetime(df_sales['Fecha'], format='%d/%m/%Y %H:%M:%S', errors='coerce')

# 4. Aggregate: collapse to one row per unique order
# If an order has 3 products, we collapse it into 1 row with the Total, so revenue isn't inflated
df_model = df_sales[['Email', 'Fecha', 'Número de orden', 'Total']].drop_duplicates(subset=['Número de orden'])

print(f" Phase 1 complete. Unique orders ready to process: {df_model.shape[0]}")

Phase 1 complete. Unique orders ready to process: 2685


In [ ]:
import datetime as dt

# 1. Define "Day Zero" (the store's most recent purchase date + 1 day)
max_date = df_model['Fecha'].max() + dt.timedelta(days=1)

# 2. Group by customer (Email) and compute their vital metrics
rfm = df_model.groupby('Email').agg({
    'Fecha': lambda x: (max_date - x.max()).days,  # Recency: days since their last purchase
    'Número de orden': 'count',                    # Frequency: total number of purchases
    'Total': 'sum'                                  # Monetary: total money spent with the brand
}).reset_index()

# 3. Rename columns
rfm.rename(columns={
    'Fecha': 'Recency_Days',
    'Número de orden': 'Frequency',
    'Total': 'Total_Monetary'
}, inplace=True)

print(rfm.head())
print(f"\n Phase 2 complete. Unique customers profiled: {rfm.shape[0]}")

                      Email  Recency_Days  Frequency  Total_Monetary
0  cliente_0001@ejemplo.com              1          1        68165.00
1  cliente_0002@ejemplo.com              1          8      1086498.99
2  cliente_0003@ejemplo.com              1          1        63165.00
3  cliente_0004@ejemplo.com              1          1        62353.00
4  cliente_0005@ejemplo.com              2          1       140857.99

 Phase 2 complete. Unique customers profiled: 2050


In [ ]:
!pip install lifetimes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 584.2/584.2 kB 12.9 MB/s eta 0:00:00


In [ ]:
from lifetimes import BetaGeoFitter
from lifetimes.utils import summary_data_from_transaction_data

# 1. Transform: adapt the data to the strict format the algorithm requires
df_predictive = summary_data_from_transaction_data(
    df_model,
    'Email',
    'Fecha',
    monetary_value_col='Total',
    observation_period_end=df_model['Fecha'].max()
)

# 2. Train: instantiate the BG/NBD model and teach it the historical purchase behavior
bgf = BetaGeoFitter(penalizer_coef=0.0)
bgf.fit(df_predictive['frequency'], df_predictive['recency'], df_predictive['T'])

# 3. Predict: ask the model to project exactly 180 days (6 months) into the future
prediction_horizon = 180
df_predictive['Predicted_Purchases_6m'] = bgf.conditional_expected_number_of_purchases_up_to_time(
    prediction_horizon,
    df_predictive['frequency'],
    df_predictive['recency'],
    df_predictive['T']
)

# 4. Business decision: rank customers from highest to lowest purchase probability
top_customers = df_predictive.sort_values(by='Predicted_Purchases_6m', ascending=False)

print("--- Top 5 Predicted Customers (Next 6 months) ---")
print(top_customers[['frequency', 'recency', 'T', 'Predicted_Purchases_6m']].head())

--- Top 5 Predicted Customers (Next 6 months) ---
                          frequency  recency      T  Predicted_Purchases_6m
Email                                                                     
cliente_0002@ejemplo.com        7.0    167.0  167.0                 3.174618
cliente_0170@ejemplo.com       13.0    607.0  640.0                 2.380183
cliente_0247@ejemplo.com       10.0    205.0  260.0                 2.269568
cliente_0162@ejemplo.com        5.0    173.0  204.0                 1.730866
cliente_0113@ejemplo.com        3.0     28.0   49.0                 1.729720


In [ ]:
from lifetimes import GammaGammaFitter

# 1. Math filter: the monetary algorithm can only learn from customers who bought more than once
recurring_customers = df_predictive[df_predictive['frequency'] > 0].copy()

# 2. Monetary training: instantiate the Gamma-Gamma model
ggf = GammaGammaFitter(penalizer_coef=0.0)
ggf.fit(recurring_customers['frequency'], recurring_customers['monetary_value'])

# 3. Predict average order value: estimate how much each customer will spend on average per future purchase
recurring_customers['Predicted_Avg_Order_Value'] = ggf.conditional_expected_average_profit(
    recurring_customers['frequency'],
    recurring_customers['monetary_value']
)

# 4. The Holy Grail (6-month LTV): multiply predicted future purchases by predicted average order value
recurring_customers['Predicted_LTV_6m'] = recurring_customers['Predicted_Purchases_6m'] * recurring_customers['Predicted_Avg_Order_Value']

# 5. Business decision: rank the file by the most profitable customers
top_ltv = recurring_customers.sort_values(by='Predicted_LTV_6m', ascending=False)

print("--- Top 5 Most Profitable Customers (Next 6 months) ---")
print(top_ltv[['frequency', 'monetary_value', 'Predicted_Purchases_6m', 'Predicted_LTV_6m']].head())

--- Top 5 Most Profitable Customers (Next 6 months) ---
                          frequency  monetary_value  Predicted_Purchases_6m  Predicted_LTV_6m
Email                                                                                        
cliente_0247@ejemplo.com       10.0   288511.900000                2.269568     636606.735251
cliente_0002@ejemplo.com        7.0   128706.998571                3.174618     409109.129711
cliente_0394@ejemplo.com       15.0   230315.734000                1.706390     387159.953317
cliente_0158@ejemplo.com        6.0   214500.000000                1.555274     322952.851532
cliente_0170@ejemplo.com       13.0   113300.614615                2.380183     271342.767374


In [ ]:
# 1. Round the numeric columns to 2 decimals
columns_to_round = {
    'monetary_value': 2,
    'Predicted_Purchases_6m': 2,
    'Predicted_Avg_Order_Value': 2,
    'Predicted_LTV_6m': 2
}
recurring_customers = recurring_customers.round(columns_to_round)

# 2. Update the Top 5 with the rounded data
top_ltv = recurring_customers.sort_values(by='Predicted_LTV_6m', ascending=False)

print("--- Top 5 Most Profitable Customers (Clean Data) ---")
print(top_ltv[['frequency', 'monetary_value', 'Predicted_Purchases_6m', 'Predicted_LTV_6m']].head())

# 3. Export the final file
top_ltv.to_csv('LTV_Prediction_DenimWest.csv', index=True, encoding='utf-8')
print("\n File 'LTV_Prediction_DenimWest.csv' exported successfully.")

--- Top 5 Most Profitable Customers (Clean Data) ---
                          frequency  monetary_value  Predicted_Purchases_6m  Predicted_LTV_6m
Email                                                                                        
cliente_0247@ejemplo.com       10.0       288511.90                    2.27         636606.74
cliente_0002@ejemplo.com        7.0       128707.00                    3.17         409109.13
cliente_0394@ejemplo.com       15.0       230315.73                    1.71         387159.95
cliente_0158@ejemplo.com        6.0       214500.00                    1.56         322952.85
cliente_0170@ejemplo.com       13.0       113300.61                    2.38         271342.77

 File 'LTV_Prediction_DenimWest.csv' exported successfully.
